In [38]:
from pathlib import Path
import shutil
from zipfile import ZipFile
import re

In [3]:
ROOT_PATH = Path("/home/daniel/code/selector_examen")
Path.mkdir(Path("examenes_quimica"), exist_ok=True)

In [4]:
def extract_files_from_zip(zipfile: Path) -> None:
    extract_path = Path(ROOT_PATH / f"examenes_quimica/{zipfile.stem}")
    with ZipFile(zipfile, "r") as file:
        file.extractall(extract_path)
    criteria_files = [i for i in extract_path.rglob("*.pdf") if "criterios" in i.name.lower()]
    for f in criteria_files:
        f.unlink(missing_ok=True)

In [96]:
def clasificar_examen(file: Path) -> str | None:

    folder = file.parent.name
    start_position = file.name.find(folder) +1
    text_to_scan = file.name[start_position:].lower()

    parts_added = []
    parts = {"extra" : "extraordinaria",
             "ord" : "ordinaria",
             "titular" : "titular",
             "suplente" : "suplente",
             "reserva a" : "reserva A",
             "reserva b" : "reserva B",
             "junio": "junio",
             "sept": "septiembre"}
    
    for k, v in parts.items():
        if k in text_to_scan:
            parts_added.append(v)

    if parts_added:
        return folder + " - " + " - ".join(parts_added) 
    else:
        return file.name

### Prueba con los exámenes de 2026

In [108]:
for year in range(2010, 2027):
    extract_files_from_zip(Path(f"quimica_zip_files/{year}_quimica.zip"))

Extraer archivos de subcarpetas

In [109]:
for folder in (ROOT_PATH / "examenes_quimica").iterdir():
    if folder.is_dir():
        # find pdf files
        pdf_files = [f for f in folder.rglob("*.pdf")]

        for f in pdf_files:

            # get subfolders of pdf files
            subfolder = str(f.relative_to(folder).parent).replace("/", " - ")

            # move them into folder root
            dest = folder/ f"{subfolder} - {f.name}"

            # move the file to folder base
            shutil.move(f, dest)
            
        # remove subfolders
        for dir in folder.iterdir():
                if dir.is_dir():
                    shutil.rmtree(dir)

In [110]:
replacements = ["exámenes química - a y b", "química", "quimica", "qu╓mica", "examen",
                "andalucía", "-a-b-cabe-anda" ]
for file in (ROOT_PATH / "examenes_quimica").rglob("*.pdf"):
    new_file = file.name.lower()

    # remove file if it is a criteria file
    if "criterio" in file.name.lower() or "_c_" in file.name.lower():
        file.unlink(missing_ok=True)
        continue

    # reemplazos
    for r in replacements:
        new_file = new_file.replace(r, "")
    
    new_file = re.sub(r"sel_\d{4}_", "", new_file)

    year_subject = " - ".join(file.parts[-2].split("_"))
    new_file = new_file.replace("_", " ").replace("-", " ")
    new_file = year_subject + " - " + new_file

    # remove extra spaces
    new_file = re.sub(r"\s+", " ", new_file)

    shutil.move(file, file.parent / f"{new_file}")

In [111]:
for folder in (ROOT_PATH / "examenes_quimica").iterdir():
    if folder.is_dir():
        file_counter = 2
        filenames = []
        for file in folder.rglob("*.pdf"):
            new_filename = clasificar_examen(file)
            if new_filename in filenames:
                filenames.append(new_filename[:-4] + f"_{file_counter}.pdf")
            else:
                filenames.append(new_filename)

    shutil.move(file, file.parent / Path(new_filename))

In [112]:
with open("archivos_examenes.txt", "w") as file:
    for f in (ROOT_PATH / "examenes_quimica").rglob("*.pdf"):
        file.write(f"{"/".join(f.parts[6:])}\n")
